# GPT-2 Fine-Tuning — Direct Comparison with Emma

## Project Overview

This notebook fine-tunes **GPT-2 Small (117M parameters)** on the exact same conversational datasets used to train Emma from scratch. The goal is a direct comparison: what happens when you start from a pre-trained language model instead of random weights?

GPT-2 was pre-trained by OpenAI on WebText — a large corpus of web pages — so it already understands grammar, facts, and language structure. Fine-tuning adapts that existing knowledge to our specific conversational task.

To make this feasible on consumer hardware (RTX 2070, 8GB VRAM), we use **LoRA (Low-Rank Adaptation)** — a technique that freezes the original model weights and injects small trainable matrices into the attention layers. Instead of training all 117M parameters, we train roughly ~1.2M (~1% of the model).

| Property | Emma (from scratch) | GPT-2 Fine-Tuned |
|---|---|---|
| Base parameters | ~48M (random init) | ~117M (pre-trained) |
| Trainable parameters | ~48M (100%) | ~1.2M (~1%) |
| Training epochs | 25 | 2 |
| Training approach | Full training from scratch | LoRA fine-tuning |

## Environment Setup

Install the required libraries. This only needs to be run once per container.

In [1]:
!pip install transformers datasets peft accelerate bitsandbytes sentencepiece ftfy -q

## Imports, Seeds, and Hardware Configuration

Same reproducibility approach as the Emma notebook — all random number generators are seeded with the same value.

In [2]:
import io
import json
import math
import os
import random
import re
import tempfile
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

import ftfy
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    DataCollatorForLanguageModeling,
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


## GPU Check

Verify that the GPU is available and check VRAM. Fine-tuning GPT-2 with LoRA requires at least 6GB of VRAM. The RTX 2070 with 8GB should have enough headroom with `fp16` enabled and a small batch size.

In [3]:
if torch.cuda.is_available():
    DEVICE = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9    
    print(f'GPU detected: {gpu_name}')
    print(f'VRAM available: {vram_gb:.1f} GB')
else:
    DEVICE = 'cpu'
    print('No GPU detected — training on CPU will be extremely slow.')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA version: {torch.version.cuda}')
print(f'Device: {DEVICE}')

GPU detected: NVIDIA GeForce RTX 2070
VRAM available: 8.6 GB
PyTorch version: 2.11.0+cu130
CUDA version: 13.0
Device: cuda


## Configuration

All tunable parameters in one place. The dataset parameters are identical to the Emma notebook for a fair comparison. The training parameters are adjusted for fine-tuning — fewer epochs, smaller batch size (to fit in 8GB VRAM), and gradient accumulation to compensate.

In [4]:
CONFIG = {
    # Dataset balance (same as Emma)
    'max_ultrachat_conversations': 175000,

    # Text filtering (same as Emma)
    'min_turn_chars': 2,
    'min_reply_chars': 2,

    # Sequence construction
    'max_seq_len': 512,

    # LoRA configuration
    'lora_r': 16,            # rank of the low-rank matrices
    'lora_alpha': 32,        # scaling factor (alpha / r = scaling)
    'lora_dropout': 0.05,    # dropout on LoRA layers

    # Training
    'batch_size': 4,         # small to fit in 8GB VRAM
    'grad_accumulation': 8,  # effective batch size = 4 * 8 = 32
    'epochs': 2,
    'learning_rate': 2e-4,
    'warmup_steps': 200,
    'fp16': True,            # half precision — essential for 8GB VRAM

    # Generation
    'temperature': 0.8,
    'top_k': 40,
    'max_new_tokens': 80,
    'repetition_penalty': 1.15,
}

PROJECT_DIR = Path.cwd()
CACHE_DIR = PROJECT_DIR / 'dataset_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('Current configuration:')
for key, value in CONFIG.items():
    print(f'  {key:>28}: {value}')

Current configuration:
   max_ultrachat_conversations: 175000
                min_turn_chars: 2
               min_reply_chars: 2
                   max_seq_len: 512
                        lora_r: 16
                    lora_alpha: 32
                  lora_dropout: 0.05
                    batch_size: 4
             grad_accumulation: 8
                        epochs: 2
                 learning_rate: 0.0002
                  warmup_steps: 200
                          fp16: True
                   temperature: 0.8
                         top_k: 40
                max_new_tokens: 80
            repetition_penalty: 1.15


## Dataset Utility Functions

These are copied directly from the Emma notebook — the same download, caching, and data loading infrastructure.

In [5]:
def download_bytes(url: str, cache_path: Path | None = None, force: bool = False) -> bytes:
    if cache_path is not None and cache_path.exists() and not force:
        return cache_path.read_bytes()

    with urllib.request.urlopen(url) as response:
        data = response.read()

    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cache_path.write_bytes(data)

    return data


def read_parquet_from_url(url: str, cache_path: Path | None = None, force: bool = False) -> pd.DataFrame:
    data = download_bytes(url, cache_path=cache_path, force=force)
    return pd.read_parquet(io.BytesIO(data))


def coerce_list(value) -> list:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if hasattr(value, 'tolist'):
        converted = value.tolist()
        if isinstance(converted, (list, tuple)):
            return list(converted)
    try:
        if hasattr(value, '__iter__') and not isinstance(value, (str, bytes, dict)):
            return list(value)
    except Exception:
        pass
    return []


def limit_conversations(conversations: list[dict], dataset_name: str, max_count: int | None = None) -> list[dict]:
    total = len(conversations)
    if max_count is None or total <= max_count:
        print(f'[{dataset_name}] Keeping all {total:,} conversations.')
        return conversations

    rng = random.Random(SEED)
    limited = conversations.copy()
    rng.shuffle(limited)
    limited = limited[:max_count]
    print(f'[{dataset_name}] Limited from {total:,} to {len(limited):,} conversations.')
    return limited


def print_dataset_preview(dataset_name: str, conversations: list[dict], n_turns: int = 4) -> None:
    print(f'[{dataset_name}] Total conversations: {len(conversations):,}')
    if not conversations:
        print(f'[{dataset_name}] No conversations loaded.')
        return

    sample = conversations[0]
    print(f'[{dataset_name}] Sample conversation id: {sample["conv_id"]}')
    for i, turn in enumerate(sample['turns'][:n_turns]):
        speaker = 'USER' if i % 2 == 0 else 'ASSISTANT'
        print(f'  {speaker}: {turn}')

## Load Datasets

Loading all four datasets identically to the Emma notebook: DailyDialog, Wizard of Wikipedia, Empathetic Dialogues, and UltraChat 200k.

In [6]:
def load_daily_dialog() -> list[dict]:
    base_url = (
        'https://huggingface.co/datasets/roskoN/dailydialog'
        '/resolve/refs%2Fconvert%2Fparquet/full'
    )

    frames = {}
    for split in ('train', 'validation', 'test'):
        url = f'{base_url}/{split}/0000.parquet'
        cache_path = CACHE_DIR / 'dailydialog' / f'{split}.parquet'
        print(f'[DailyDialog] Fetching {split}...')
        frames[split] = read_parquet_from_url(url, cache_path=cache_path)
        print(f'[DailyDialog] {split} rows: {len(frames[split]):,}')

    conversations = []
    for split_name, df in frames.items():
        for row_index, row in df.iterrows():
            turns = [str(turn).strip() for turn in coerce_list(row.get('utterances', [])) if str(turn).strip()]
            if len(turns) >= 2:
                conversations.append({
                    'source': 'daily_dialog',
                    'split': split_name,
                    'conv_id': f'dd_{split_name}_{row_index}',
                    'turns': turns,
                })

    print_dataset_preview('DailyDialog', conversations)
    return conversations


daily_dialog_conversations = load_daily_dialog()

[DailyDialog] Fetching train...
[DailyDialog] train rows: 11,118
[DailyDialog] Fetching validation...
[DailyDialog] validation rows: 1,000
[DailyDialog] Fetching test...
[DailyDialog] test rows: 1,000
[DailyDialog] Total conversations: 13,118
[DailyDialog] Sample conversation id: dd_train_0
  USER: Say , Jim , how about going for a few beers after dinner ?
  ASSISTANT: You know that is tempting but is really not good for our fitness .
  USER: What do you mean ? It will help us to relax .
  ASSISTANT: Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?


In [7]:
def clean_wizard_opening_post(post_text: str, topic_text: str) -> tuple[bool, str]:
    post_text = str(post_text).strip()
    topic_text = str(topic_text).strip()

    if not post_text:
        return False, ''

    if topic_text:
        if post_text == topic_text:
            return False, ''
        topic_prefix = f'{topic_text}\n'
        if post_text.startswith(topic_prefix):
            post_text = post_text[len(topic_prefix):].strip()

    return bool(post_text), post_text


def load_wizard_of_wikipedia() -> list[dict]:
    base_url = (
        'https://huggingface.co/datasets/chujiezheng/wizard_of_wikipedia'
        '/resolve/refs%2Fconvert%2Fparquet/default'
    )

    frames = {}
    for split in ('train', 'validation', 'test'):
        url = f'{base_url}/{split}/0000.parquet'
        cache_path = CACHE_DIR / 'wizard_of_wikipedia' / f'{split}.parquet'
        print(f'[WizardOfWikipedia] Fetching {split}...')
        frames[split] = read_parquet_from_url(url, cache_path=cache_path)
        print(f'[WizardOfWikipedia] {split} rows: {len(frames[split]):,}')

    conversations = []
    for split_name, df in frames.items():
        for row_index, row in df.iterrows():
            posts = coerce_list(row.get('post', []))
            responses = coerce_list(row.get('response', []))
            topics = coerce_list(row.get('topics', []))
            topic_text = str(topics[0]).strip() if topics else ''

            turns = []
            for turn_index, (post_text, response_text) in enumerate(zip(posts, responses)):
                if turn_index == 0:
                    keep_post, cleaned_post = clean_wizard_opening_post(post_text, topic_text)
                else:
                    cleaned_post = str(post_text).strip()
                    keep_post = bool(cleaned_post)

                cleaned_response = str(response_text).strip()
                if not keep_post or not cleaned_response:
                    continue

                turns.append(cleaned_post)
                turns.append(cleaned_response)

            if len(turns) >= 2:
                conversations.append({
                    'source': 'wizard_of_wikipedia',
                    'split': split_name,
                    'conv_id': f'wow_{split_name}_{row_index}',
                    'turns': turns,
                })

    print_dataset_preview('WizardOfWikipedia', conversations)
    return conversations


wow_conversations = load_wizard_of_wikipedia()

[WizardOfWikipedia] Fetching train...
[WizardOfWikipedia] train rows: 18,430
[WizardOfWikipedia] Fetching validation...
[WizardOfWikipedia] validation rows: 1,948
[WizardOfWikipedia] Fetching test...
[WizardOfWikipedia] test rows: 1,933
[WizardOfWikipedia] Total conversations: 22,311
[WizardOfWikipedia] Sample conversation id: wow_train_0
  USER: I'm a huge fan of science fiction myself!
  ASSISTANT: Awesome! I really love how sci-fi storytellers focus on political/social/philosophical issues that would still be around even in the future. Makes them relatable.
  USER: I agree. One of my favorite forms of science fiction is anything related to time travel! I find it fascinating.
  ASSISTANT: It's not quite sci-fi, but my favorite version of time travel is in Harry Potter and the Prisoner of Azkaban. Breaks zero logical rules.


In [8]:
def load_empathetic_dialogues() -> list[dict]:
    base_url = (
        'https://huggingface.co/datasets/Estwld/empathetic_dialogues_llm'
        '/resolve/refs%2Fconvert%2Fparquet/default'
    )

    frames = {}
    for split in ('train', 'valid', 'test'):
        url = f'{base_url}/{split}/0000.parquet'
        cache_path = CACHE_DIR / 'empathetic_dialogues' / f'{split}.parquet'
        print(f'[EmpatheticDialogues] Fetching {split}...')
        frames[split] = read_parquet_from_url(url, cache_path=cache_path)
        print(f'[EmpatheticDialogues] {split} rows: {len(frames[split]):,}')

    conversations = []
    for split_name, df in frames.items():
        for row_index, row in df.iterrows():
            messages = row.get('conversations', [])
            turns = []
            for message in messages:
                if not isinstance(message, dict):
                    continue
                content = str(message.get('content', '')).strip()
                if content:
                    turns.append(content)

            if len(turns) >= 2:
                conversations.append({
                    'source': 'empathetic_dialogues',
                    'split': split_name,
                    'conv_id': f'ed_{split_name}_{row_index}',
                    'turns': turns,
                })

    print_dataset_preview('EmpatheticDialogues', conversations)
    return conversations


empathetic_conversations = load_empathetic_dialogues()

[EmpatheticDialogues] Fetching train...
[EmpatheticDialogues] train rows: 19,533
[EmpatheticDialogues] Fetching valid...
[EmpatheticDialogues] valid rows: 2,770
[EmpatheticDialogues] Fetching test...
[EmpatheticDialogues] test rows: 2,547
[EmpatheticDialogues] Total conversations: 24,847
[EmpatheticDialogues] Sample conversation id: ed_train_0
  USER: I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.
  ASSISTANT: Was this a friend you were in love with, or just a best friend?
  USER: This was a best friend. I miss her.
  ASSISTANT: Where has she gone?


In [9]:
def load_ultrachat() -> list[dict]:
    base_url = (
        'https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k'
        '/resolve/refs%2Fconvert%2Fparquet/default'
    )

    frames = {}
    for split in ('train_sft', 'test_sft'):
        shard_frames = []
        shard = 0
        while True:
            url = f'{base_url}/{split}/{shard:04d}.parquet'
            cache_path = CACHE_DIR / 'ultrachat_200k' / f'{split}_{shard:04d}.parquet'
            try:
                print(f'[UltraChat200k] Fetching {split} shard {shard}...')
                df = read_parquet_from_url(url, cache_path=cache_path)
                df['_shard'] = shard
                shard_frames.append(df)
                print(f'[UltraChat200k] {split} shard {shard}: {len(df):,} rows')
                shard += 1
            except Exception:
                print(f'[UltraChat200k] No more shards for {split} (stopped at shard {shard})')
                break

        if not shard_frames:
            frames[split] = pd.DataFrame()
        else:
            frames[split] = pd.concat(shard_frames, ignore_index=True)
            print(f'[UltraChat200k] {split} total rows: {len(frames[split]):,}')

    conversations = []
    for split_name, df in frames.items():
        if df.empty:
            continue
        for row_index, row in df.iterrows():
            messages = coerce_list(row.get('messages', []))
            role_content_pairs = []

            for message in messages:
                if not isinstance(message, dict):
                    continue
                role = str(message.get('role', '')).strip().lower()
                content = message.get('content', '')
                if role not in ('user', 'assistant'):
                    continue
                if not isinstance(content, str):
                    continue
                content = content.strip()
                if not content:
                    continue
                role_content_pairs.append((role, content))

            while role_content_pairs and role_content_pairs[0][0] != 'user':
                role_content_pairs.pop(0)

            turns = []
            expected_role = 'user'
            for role, content in role_content_pairs:
                if role != expected_role:
                    continue
                turns.append(content)
                expected_role = 'assistant' if expected_role == 'user' else 'user'

            if len(turns) % 2 != 0:
                turns = turns[:-1]

            if len(turns) >= 2:
                shard_num = int(row.get('_shard', 0))
                conversations.append({
                    'source': 'ultrachat200k',
                    'split': split_name,
                    'conv_id': f'uc_{split_name}_s{shard_num}_{row_index}',
                    'turns': turns,
                })

    conversations = limit_conversations(
        conversations,
        dataset_name='UltraChat200k',
        max_count=CONFIG['max_ultrachat_conversations'],
    )
    print_dataset_preview('UltraChat200k', conversations)
    return conversations


ultrachat_conversations = load_ultrachat()

[UltraChat200k] Fetching train_sft shard 0...
[UltraChat200k] train_sft shard 0: 69,289 rows
[UltraChat200k] Fetching train_sft shard 1...
[UltraChat200k] train_sft shard 1: 69,288 rows
[UltraChat200k] Fetching train_sft shard 2...
[UltraChat200k] train_sft shard 2: 69,288 rows
[UltraChat200k] Fetching train_sft shard 3...
[UltraChat200k] No more shards for train_sft (stopped at shard 3)
[UltraChat200k] train_sft total rows: 207,865
[UltraChat200k] Fetching test_sft shard 0...
[UltraChat200k] test_sft shard 0: 23,110 rows
[UltraChat200k] Fetching test_sft shard 1...
[UltraChat200k] No more shards for test_sft (stopped at shard 1)
[UltraChat200k] test_sft total rows: 23,110
[UltraChat200k] Limited from 230,975 to 175,000 conversations.
[UltraChat200k] Total conversations: 175,000
[UltraChat200k] Sample conversation id: uc_train_sft_s2_194737
  USER: Monitor the content for engagement and feedback by analyzing the number of views, likes, comments, shares, and click-through rates. Assess 

## Text Normalisation

Same cleaning pipeline as Emma — fix Unicode, normalise whitespace, filter out low-quality turns.

In [10]:
_whitespace_re = re.compile(r"\s+")
_non_text_re = re.compile(r"[^a-zA-Z0-9.!?,;:()\[\]{}\-\s'\"/]+")


def normalize_text(text: str) -> str:
    text = ftfy.fix_text(str(text))
    text = unicodedata.normalize('NFKC', text)
    text = (
        text.replace('\u2019', "'")
            .replace('\u2018', "'")
            .replace('\u201c', '"')
            .replace('\u201d', '"')
    )
    text = _non_text_re.sub(' ', text)
    text = _whitespace_re.sub(' ', text).strip()
    return text


def is_good_turn(text: str, min_chars: int) -> bool:
    if not text:
        return False
    stripped = text.strip()
    if len(stripped) < min_chars:
        return False
    if not re.search(r'[a-zA-Z0-9]', stripped):
        return False
    return True


all_conversations = (
    daily_dialog_conversations
    + wow_conversations
    + empathetic_conversations
    + ultrachat_conversations
)

print(f'Total raw conversations before normalization: {len(all_conversations):,}')
normalized_conversations = []

for index, conversation in enumerate(all_conversations, start=1):
    normalized_turns = [normalize_text(turn) for turn in conversation['turns']]
    normalized_turns = [turn for turn in normalized_turns if is_good_turn(turn, CONFIG['min_turn_chars'])]

    if len(normalized_turns) >= 2 and len(normalized_turns) % 2 != 0:
        normalized_turns = normalized_turns[:-1]

    if len(normalized_turns) >= 2:
        normalized_conversations.append({**conversation, 'turns': normalized_turns})

    if index % 10000 == 0:
        print(f'  normalized {index:,} / {len(all_conversations):,} conversations...')

print(f'Combined normalized conversations: {len(normalized_conversations):,}')
source_counts = pd.Series([c['source'] for c in normalized_conversations]).value_counts()
display(source_counts.to_frame('count'))

Total raw conversations before normalization: 235,276
  normalized 10,000 / 235,276 conversations...
  normalized 20,000 / 235,276 conversations...
  normalized 30,000 / 235,276 conversations...
  normalized 40,000 / 235,276 conversations...
  normalized 50,000 / 235,276 conversations...
  normalized 60,000 / 235,276 conversations...
  normalized 70,000 / 235,276 conversations...
  normalized 80,000 / 235,276 conversations...
  normalized 90,000 / 235,276 conversations...
  normalized 100,000 / 235,276 conversations...
  normalized 110,000 / 235,276 conversations...
  normalized 120,000 / 235,276 conversations...
  normalized 130,000 / 235,276 conversations...
  normalized 140,000 / 235,276 conversations...
  normalized 150,000 / 235,276 conversations...
  normalized 160,000 / 235,276 conversations...
  normalized 170,000 / 235,276 conversations...
  normalized 180,000 / 235,276 conversations...
  normalized 190,000 / 235,276 conversations...
  normalized 200,000 / 235,276 conversation

,count
ultrachat200k,175000
empathetic_dialogues,24846
wizard_of_wikipedia,22311
daily_dialog,13118


## Train / Validation Split

Same 80/10/10 split as Emma.

In [11]:
shuffled_conversations = normalized_conversations.copy()
random.shuffle(shuffled_conversations)

n_total = len(shuffled_conversations)
train_end = math.floor(n_total * 0.80)
val_end = math.floor(n_total * 0.90)

train_conversations = shuffled_conversations[:train_end]
val_conversations = shuffled_conversations[train_end:val_end]
test_conversations = shuffled_conversations[val_end:]

print(f'Total conversations: {n_total:,}')
print(f'  train: {len(train_conversations):,} ({len(train_conversations) / n_total:.1%})')
print(f'  val:   {len(val_conversations):,} ({len(val_conversations) / n_total:.1%})')
print(f'  test:  {len(test_conversations):,} ({len(test_conversations) / n_total:.1%})')

Total conversations: 235,275
  train: 188,220 (80.0%)
  val:   23,527 (10.0%)
  test:  23,528 (10.0%)


## Load GPT-2 Tokeniser and Format Conversations

Instead of training a custom SentencePiece tokeniser like Emma, we use GPT-2's built-in BPE tokeniser with a vocabulary of 50,257 tokens. The conversation format is a simple plaintext layout — GPT-2 does not need custom special tokens because it was pre-trained on natural text.

In [12]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token


def format_conversation(conversation: dict) -> str:
    turns = conversation['turns']
    text = ''
    for i, turn in enumerate(turns):
        role = 'User' if i % 2 == 0 else 'Assistant'
        text += f'{role}: {turn}\n'
    text += tokenizer.eos_token
    return text


# Sanity check
sample = format_conversation(train_conversations[0])
print('Formatted sample:')
print(sample[:500])
print(f'\nTokenised length: {len(tokenizer.encode(sample))} tokens')
print(f'GPT-2 vocabulary size: {len(tokenizer):,}')

Formatted sample:
User: I like heavy metal as well. Which metal bands do you like?
Assistant: One of my favorites is Sabaton. Metal music is largely from the UK which is strange to me.
User: Yeah think Ozzy Osbourne haha. He is the Godfather of Metal.
Assistant: Is he? Cool! Metal is quite new too, since it was made in the late 1960s.
<|endoftext|>

Tokenised length: 85 tokens
GPT-2 vocabulary size: 50,257


## Tokenise the Dataset

Format all conversations into plain text strings, tokenise them, and wrap them in HuggingFace `Dataset` objects for the `Trainer` API.

In [ ]:
def tokenize(example):
    return tokenizer(
        example['text'],
        truncation=True,
        max_length=CONFIG['max_seq_len'],
        padding='max_length',
    )


train_texts = [format_conversation(c) for c in train_conversations]
val_texts = [format_conversation(c) for c in val_conversations]

print(f'Formatting complete. Tokenising...')

train_dataset = Dataset.from_dict({'text': train_texts}).map(
    tokenize, batched=True, batch_size=1000
)
# Free memory before processing validation set
del train_texts
import gc
gc.collect()

val_dataset = Dataset.from_dict({'text': val_texts}).map(
    tokenize, batched=True, batch_size=1000
)
del val_texts
gc.collect()

print(f'Train examples: {len(train_dataset):,}')
print(f'Val examples:   {len(val_dataset):,}')

Formatting complete. Tokenising...


## Load GPT-2 and Apply LoRA

Load the pre-trained GPT-2 Small model (117M parameters) and apply LoRA. This freezes all original weights and injects small trainable low-rank matrices into the attention projection layers. The result is that only ~1% of the parameters are actually updated during training, making it feasible on consumer hardware.

In [ ]:
model = GPT2LMHeadModel.from_pretrained('gpt2')

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=['c_attn', 'c_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training

Fine-tune the model using HuggingFace's `Trainer` API. Key differences from Emma's training:

| Aspect | Emma | GPT-2 Fine-Tune |
|---|---|---|
| Epochs | 25 | 2 |
| Batch size | 128 | 4 (×8 accumulation = effective 32) |
| Precision | mixed_float16 | fp16 |
| Parameters trained | 48M (100%) | ~1.2M (~1%) |

In [ ]:
training_args = TrainingArguments(
    output_dir='./gpt2_finetuned',
    num_train_epochs=CONFIG['epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['grad_accumulation'],
    learning_rate=CONFIG['learning_rate'],
    warmup_steps=CONFIG['warmup_steps'],
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=CONFIG['fp16'],
    seed=SEED,
    logging_steps=100,
    report_to='none',
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print(f'Training on {len(train_dataset):,} examples')
print(f'Batch size: {CONFIG["batch_size"]} x {CONFIG["grad_accumulation"]} accumulation = {CONFIG["batch_size"] * CONFIG["grad_accumulation"]} effective')
print(f'Epochs: {CONFIG["epochs"]}')
print(f'fp16: {CONFIG["fp16"]}')
print()

trainer.train()

## Save the Fine-Tuned Model

Save the LoRA adapter weights and the tokeniser so the model can be reloaded later without retraining.

In [ ]:
SAVE_DIR = PROJECT_DIR / 'gpt2_finetuned_final'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Model saved to: {SAVE_DIR.resolve()}')

## Inference — Same Test Prompts as Emma

Run the exact same test prompts used to evaluate Emma for a direct side-by-side comparison.

In [ ]:
model.eval()
model.to(DEVICE)


def generate_reply(conversation_turns: list[str], max_new_tokens: int = 80) -> str:
    prompt = ''
    for i, turn in enumerate(conversation_turns):
        role = 'User' if i % 2 == 0 else 'Assistant'
        prompt += f'{role}: {turn}\n'
    prompt += 'Assistant:'

    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=CONFIG['temperature'],
            top_k=CONFIG['top_k'],
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=CONFIG['repetition_penalty'],
        )

    reply_ids = outputs[0][inputs['input_ids'].shape[1]:]
    reply = tokenizer.decode(reply_ids, skip_special_tokens=True).strip()

    # Clean up: stop at the next 'User:' if the model generates one
    if 'User:' in reply:
        reply = reply[:reply.index('User:')].strip()

    return reply


test_prompts = [
    'What are the benefits of regular exercise?',
    'Can you suggest three popular tourist destinations for a summer vacation?',
    'Explain the concept of photosynthesis in simple terms.',
    'Tell me a fun fact about animals.',
    'Give me a short positive affirmation for the day.',
    'How can I improve my problem-solving skills?',
]

conversation_turns = []
for prompt in test_prompts:
    print('\u2500' * 60)
    print(f'You:  {prompt}')
    reply = generate_reply(
        conversation_turns + [prompt],
        max_new_tokens=CONFIG['max_new_tokens'],
    )
    print(f'Bot:  {reply}')
    conversation_turns.append(prompt)
    conversation_turns.append(reply)

print('\u2500' * 60)